# Tập 3.6 · SLAM: con gà hay quả trứng? 🐔🥚

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap36/tap36_slam.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~45 phút · Giáo trình: Principles of Robot Autonomy, [Chương 14](https://porabook.com/book/slam/)

Vừa định vị vừa vẽ bản đồ: Graph SLAM với một cạnh khép vòng. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** phép ghép tư thế, lõi của ICP và sai số cạnh của đồ thị (3 bài), có ô chấm tự động.
3. Vẽ bản đồ trước và sau khép vòng bằng hàm của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap36")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import slam as mp                  # mô phỏng trong video (không đặt tên goc vì bài làm có hàm goc)
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
e_dr, e_opt = mp.loi(mp.DR), mp.loi(mp.OPT)
print(f"Chỉ odometry: cuối lệch {e_dr[-1]:.2f} m, trung bình {e_dr.mean():.2f} m · bản đồ nét {mp.do_net(mp.G_DR) * 100:.1f}%")
print(f"Sau tối ưu đồ thị: cuối lệch {e_opt[-1]:.2f} m, trung bình {e_opt.mean():.2f} m · bản đồ nét {mp.do_net(mp.G_OPT) * 100:.1f}%")

## 2. Bài tập

$a \oplus b = [a_x + b_x\cos a_\theta - b_y\sin a_\theta,\ a_y + b_x\sin a_\theta + b_y\cos a_\theta,\ a_\theta + b_\theta]$; sai số cạnh $r_{ij} = (x_i^{-1}\oplus x_j) - z_{ij}$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `cong(a, b)`, `tru(a, b)` | ghép / tách tư thế 2D |
| **2** | `can_chinh(P, Q)` | phép quay + dời tốt nhất cho các cặp điểm (SVD) |
| **3** | `sai_so_canh(xi, xj, z)` | sai số một cạnh, bọc góc |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def goc(a):
    """Có sẵn: bọc góc về [−π, π)."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def cong(a, b):
    """Bài 1a: a ⊕ b. Xoay b theo hướng a[2] rồi cộng vào a:
        [a_x + cos·b_x − sin·b_y,  a_y + sin·b_x + cos·b_y,  goc(a_θ + b_θ)]"""
    raise NotImplementedError("TODO Bài 1a: ghép tư thế")


def tru(a, b):
    """Bài 1b: a⁻¹ ⊕ b, tức tư thế b nhìn trong khung của a. Với dx = b_x − a_x, dy = b_y − a_y:
        [cos·dx + sin·dy,  −sin·dx + cos·dy,  goc(b_θ − a_θ)]       (cos, sin của a_θ)"""
    raise NotImplementedError("TODO Bài 1b: tư thế tương đối")


def can_chinh(P, Q):
    """Bài 2 (bước lõi của ICP): tìm phép quay R (2×2) và dời t sao cho R·p + t ≈ q với các cặp điểm đã ghép (P[i] ↔ Q[i]).

    1. Trừ tâm: mp = trung bình P, mq = trung bình Q.
    2. H = (P − mp)ᵀ·(Q − mq);  U, S, Vt = np.linalg.svd(H);  R = Vtᵀ·Uᵀ.
    3. Nếu det(R) < 0 (bị lật gương): đổi dấu hàng cuối của Vt rồi tính lại R.
    4. t = mq − R·mp.
    Trả về (dth, t) với dth = atan2(R[1,0], R[0,0])."""
    raise NotImplementedError("TODO Bài 2: căn chỉnh hai đám điểm")


def sai_so_canh(xi, xj, z):
    """Bài 3 (Mục 14.7): sai số của một cạnh trong đồ thị. Cạnh nói "từ nút i, nút j nằm ở z".

    r = (xi⁻¹ ⊕ xj) − z, nhớ bọc thành phần góc r[2] bằng goc(). Trả về mảng 3 số."""
    raise NotImplementedError("TODO Bài 3: sai số cạnh")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1 và 3, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Cạnh khép vòng được tin cậy cỡ nào thì đủ? Chia Ω của nó cho các số khác nhau rồi tối ưu lại.

In [ ]:
import loi_giai
from kiem_tra import toi_uu
cu = mp.CANH_VONG
for chia in [1, 100, 10000]:
    mp.CANH_VONG = (cu[0], cu[1], cu[2], cu[3] / chia)
    e = mp.loi(toi_uu(loi_giai.sai_so_canh, mp.DR))
    print(f"Ω khép vòng / {chia:5d}: cuối lệch {e[-1]:.2f} m, trung bình {e.mean():.2f} m")
mp.CANH_VONG = cu

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap36).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)